# 🚀 MDAAR: Multi-role Dynamic Architecture for Automated Repositories
### AI-Driven Software Documentation Framework (Complete Interactive Pipeline)
---
**المميزات المعمارية الشاملة:**
1. **مجاني 100% وبدون اشتراكات خارجية:** يعمل بالكامل على Google Colab و Google Drive.
2. **مفاتيح SSH مستقلة تماماً:** مفتاح Deploy Key خاص بمستودع الكود، ومفتاح Deploy Key منفصل تماماً لمستودع التوثيق الخارجي.
3. **خيارات ذكاء اصطناعي مرنة ومتعددة:** دعم Gemini API (مع تدوير المفاتيح) أو نماذج محلية أوفلاين (Transformers / Local GGUF).
4. **استخلاص تلقائي لمخططات المعمارية وقواعد البيانات:** محرك Schema Parser (SQL ➔ Mermaid ERD) ومحرك Route Parser لاستخراج الـ APIs تلقائياً.
5. **منظومة وكلاء التوثيق المتعددين (Multi-Agent System):** وكيل المعمارية، قواعد البيانات، الواجهات، التشغيل، وأدلة المستخدم.
6. **التوثيق الحي المستمر (Living Documentation):** تحليل أثر التغييرات عبر `git diff` وتحديث الوثائق المتأثرة فقط.
7. **مساعد معرفي ذكي تفاعلي (Smart AI Q&A / RAG):** استجواب الوثائق باللغة الطبيعية مع استشهاد بالمصادر مجاناً 100%.
8. **بوابة MkDocs Material + نفق Cloudflare Tunnel:** معاينة الموقع التفاعلي عبر الإنترنت فوراً.

## 1️⃣ تجهيز البيئة وتثبيت MDAAR وربط Google Drive
تفعيل مساحة التخزين الدائمة، وتثبيت حزمة **MDAAR** مباشرة من GitHub مع جميع التبعيات (MkDocs Material, Google GenAI, Cloudflare Tunnel).

In [ ]:
#@title 1. تفعيل الربط وتثبيت محرك MDAAR والاعتماديات { display-mode: "form" }
import os, sys, json, shutil, subprocess
from pathlib import Path

# 1. ربط Google Drive للاستدامة
DRIVE_MOUNT_POINT = '/content/drive'
if not os.path.exists(DRIVE_MOUNT_POINT):
    try:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT)
        print('✅ تم ربط Google Drive بنجاح.')
    except Exception as e:
        print('⚠️ تنبيه: لم يتم ربط الدرايف تفاعلياً، سيتم استخدام مسار تخزين محلي مؤقتاً:', e)

# 2. تثبيت مكتبة MDAAR مع أدوات التكميم والتسريع للنماذج المحلية (GPU T4)
print('⏳ جاري تثبيت حزمة MDAAR والاعتماديات ومكتبات التكميم (accelerate, bitsandbytes)...')
subprocess.run(['pip', 'install', '-q', '--upgrade', 'git+https://github.com/braadiv/MDAAR.git'], check=True)
subprocess.run(['pip', 'install', '-q', '--upgrade', 'accelerate', 'bitsandbytes'], check=True)

# 3. التحقق من cloudflared لنفق المعاينة
cf_check = subprocess.run(['which', 'cloudflared'], capture_output=True, text=True)
if not cf_check.stdout.strip():
    print('⏳ تثبيت cloudflared...')
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], check=True)

print('🎉 تم تجهيز البيئة وتثبيت MDAAR وأدوات التكميم بنجاح!')

## 2️⃣ ضبط إعدادات المشروع ومستودع الكود (Source Repo)
حدد اسم المشروع ورابط مستودع الكود المصدري وطريقة المصادقة.
في حال اختيار **SSH**، سيتم توليد زوج مفاتيح خاص بمستودع الكود (`id_ed25519_source`) وطباعة المفتاح العام لربطه في GitHub/GitLab كـ Deploy Key.

In [ ]:
#@title 2. ضبط مستودع الكود المصدري والاتصال الآمن { display-mode: "form" }
PROJECT_NAME = "MySystem" #@param {type:"string"}
GIT_SOURCE_URL = "https://github.com/octocat/Hello-World.git" #@param {type:"string"}
SOURCE_AUTH_METHOD = "HTTPS" #@param ["HTTPS", "SSH"]

# مسار التخزين في Google Drive أو مسار العمل المحلي
if os.path.exists('/content/drive/MyDrive'):
    BASE_WORKSPACE = Path('/content/drive/MyDrive/AI_Documentation_Workspace')
else:
    BASE_WORKSPACE = Path('/content/AI_Documentation_Workspace')

from mdaar_pipeline import MDAARPipeline, ModelGateway

pipeline = MDAARPipeline(
    project_name=PROJECT_NAME,
    base_workspace=BASE_WORKSPACE,
    git_source_url=GIT_SOURCE_URL,
    source_auth=SOURCE_AUTH_METHOD
)

if SOURCE_AUTH_METHOD == "SSH":
    source_pub_key = pipeline.setup_source_ssh()
    print('='*70)
    print('🔑 مفتاح SSH العام لمستودع الكود (Source Deploy Key):')
    print('أضفه في إعدادات مستودع الكود (Settings -> Deploy Keys):')
    print(source_pub_key)
    print('='*70)

print(f'✅ مجلد المشروع جاهز في المسار:\n📁 {pipeline.project_dir}')

## 3️⃣ ضبط مستودع التوثيق الخارجي المستقل (Docs Repo - اختياري)
إذا كنت ترغب بحفظ ونشر التوثيق في مستودع GitHub/GitLab منفصل، حدد رابطه هنا.
يمكن أن يكون هذا المستودع على حساب أو منصة أخرى تماماً وله **مفتاح SSH مستقل خاص به** (`id_ed25519_docs`).

In [ ]:
#@title 3. ضبط مستودع التوثيق المنفصل والمفتاح المستقل { display-mode: "form" }
ENABLE_EXTERNAL_DOCS_REPO = False #@param {type:"boolean"}
GIT_DOCS_URL = "" #@param {type:"string"}
DOCS_AUTH_METHOD = "HTTPS" #@param ["HTTPS", "SSH"]

if ENABLE_EXTERNAL_DOCS_REPO and GIT_DOCS_URL.strip():
    pipeline.git_docs_url = GIT_DOCS_URL.strip()
    pipeline.docs_auth = DOCS_AUTH_METHOD
    
    if DOCS_AUTH_METHOD == "SSH":
        docs_pub_key = pipeline.setup_docs_ssh()
        print('='*70)
        print('🔑 مفتاح SSH العام لمستودع التوثيق (Docs Deploy Key):')
        print('أضفه في مستودع الوثائق المستقل (Settings -> Deploy Keys مع صلاحية Write):')
        print(docs_pub_key)
        print('='*70)
    print(f'✅ تم ربط مستودع التوثيق الخارجي: {GIT_DOCS_URL}')
else:
    pipeline.git_docs_url = None
    print('ℹ️ التوثيق سيُحفظ محلياً وبشكل دائم داخل Google Drive فقط.')

## 4️⃣ اختيار وإعداد محرك الذكاء الاصطناعي (Model Gateway)
اختر بين استدعاء **Google Gemini API** (سريع ومجاني بسياق ضخم وتدوير مفاتيح) أو تشغيل **النماذج المحلية القوية المكممة (4-bit NF4)** مثل:
- **Qwen2.5-Coder-7B:** الأقوى لهندسة وتوثيق الأكواد ومخططات المعمارية (مجاني ومفتوح 100% بدون توكن).
- **Meta-Llama-3.1-8B-Instruct:** ممتاز للاستنتاج التقني (يتطلب توكن Hugging Face).
- **Mistral-7B-Instruct-v0.3:** نافذة سياق واسعة (32k) وتوليد دقيق (يتطلب توكن Hugging Face).
- **Gemma-2-9B-IT:** نموذج قوي بجودة عالية من Google (يتطلب توكن Hugging Face).
> 💡 **ملاحظة الأداء على كارت T4:** تكميم 4-bit يستهلك حوالي 5.5 جيجابايت فقط من أصل 15 جيجابايت VRAM، مما يمنع التهنيج تماماً.

In [ ]:
#@title 4. اختيار وتكوين نموذج الذكاء الاصطناعي { display-mode: "form" }
AI_PROVIDER = "Gemini" #@param ["Gemini", "Local-Offline"]
GEMINI_MODEL = "gemini-2.5-flash" #@param ["gemini-2.5-flash", "gemini-1.5-pro", "gemini-2.0-flash"]
GEMINI_KEYS_INPUT = "" #@param {type:"string"}

# خيارات النماذج المحلية (متوافقة 100% مع كارت كولاب T4 بتكميم 4-bit)
LOCAL_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct" #@param ["Qwen/Qwen2.5-Coder-7B-Instruct", "meta-llama/Meta-Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-Coder-1.5B-Instruct"]
QUANTIZATION = "4bit" #@param ["4bit", "None"]
HF_TOKEN_INPUT = "" #@param {type:"string"}

# استخراج مفاتيح Gemini
keys_list = [k.strip() for k in GEMINI_KEYS_INPUT.split(',') if k.strip()]
if not keys_list:
    try:
        from google.colab import userdata
        secret_key = userdata.get('GEMINI_API_KEY')
        if secret_key:
            keys_list.append(secret_key)
    except:
        pass

# استخراج توكن Hugging Face للنماذج المغلقة (Llama, Mistral, Gemma)
hf_token = HF_TOKEN_INPUT.strip() or None
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except:
        pass

gateway = ModelGateway(
    provider=AI_PROVIDER,
    model_name=GEMINI_MODEL,
    api_keys=keys_list,
    local_model_path=LOCAL_MODEL_NAME,
    hf_token=hf_token,
    quantization=QUANTIZATION
)
pipeline.gateway = gateway

print(f'✅ تم إعداد بوابة الذكاء الاصطناعي بنجاح!')
print(f'  • المزود المعتمد: {AI_PROVIDER}')
if AI_PROVIDER == 'Gemini':
    print(f'  • النموذج: {GEMINI_MODEL} (عدد المفاتيح المسجلة لتدوير الاستهلاك: {len(keys_list)})')
else:
    print(f'  • النموذج المحلي: {LOCAL_MODEL_NAME}')
    print(f'  • نمط التكميم (Quantization): {QUANTIZATION}')

## 5️⃣ استنساخ الكود وتتبع التغييرات (Git Diff Tracker)
فحص الكود وحساب التغييرات منذ آخر عملية توثيق مسجلة في `.state.json`.

In [ ]:
#@title 5. مزامنة الكود ورصد الملفات المعدلة { display-mode: "form" }
current_commit, changed_files = pipeline.sync_source_repo()

print(f'📌 حالة الـ Commit الحالي: ({current_commit[:7] if current_commit else "Initial"})')
if changed_files:
    print(f'🔍 تم رصد {len(changed_files)} ملفاً متغيراً منذ آخر توثيق:')
    for f in changed_files[:10]:
        print('  •', f)
else:
    print('ℹ️ لا توجد تغييرات جديدة أو أن هذا التشغيل الأولي للمشروع.')

## 6️⃣ بناء شجرة التوثيق واستخلاص المخططات (Parsers & Agents)
توليد حزم التوثيق (Governance, Administrative, Technical, Operations, User Guides) وتشغيل أدوات التحليل المتخصصة لاستخراج جداول SQL ومسارات الـ APIs وتوليد الوثائق عبر الوكلاء الأذكياء.

In [ ]:
#@title 6. إنشاء الهيكل وتشغيل الوكلاء والمحللات { display-mode: "form" }
FORCE_FULL_GENERATION = True #@param {type:"boolean"}

# 1. إنشاء قوالب الهيكل وسجل الوثائق
created_count = pipeline.init_framework_structure()
print(f'✅ اكتمل تجهيز شجرة التوثيق (تم إنشاء/تحديث {created_count} ملف قالب).')

# 2. استخراج المخططات الحقيقية ومسارات الكود
pipeline.extract_and_generate_specialized()

# 3. تشغيل الوكلاء لتوليد الوثائق المعمارية والتشغيلية
target_changes = None if FORCE_FULL_GENERATION else changed_files
pipeline.run_multi_agent_generation(changed_files=target_changes)

# 4. حفظ حالة الـ Commit في .state.json
if current_commit:
    pipeline.save_state(current_commit)
    print(f'💾 تم حفظ حالة التوثيق للـ Commit ({current_commit[:7]}).')

## 7️⃣ إطلاق بوابة التوثيق التفاعلية (MkDocs Material + Cloudflare Tunnel)
بناء وتشغيل الموقع التفاعلي ومشاركته عبر نفق عام آمن وفوري، مع مزامنة مستودع الوثائق الخارجي (إن كان مفعلاً).

In [ ]:
#@title 7. تشغيل البوابة واستخراج الرابط العام { display-mode: "form" }
tunnel_url = pipeline.serve_and_tunnel(port=8000)

if tunnel_url:
    print('='*70)
    print('🎉 بوابة التوثيق التفاعلية جاهزة ومتاحة عبر الإنترنت:')
    print(f'👉 {tunnel_url}')
    print('='*70)
else:
    print('⚠️ جاري تشغيل الخادم محلياً، يمكنك التحقق من سجلات النفق.')

# رفع التوثيق إلى مستودع التوثيق الخارجي (إن كان مضبوطاً ومفعلاً)
sync_docs_res = pipeline.sync_docs_repo()
print('📡 مزامنة مستودع الوثائق:', sync_docs_res)

## 8️⃣ المساعد المعرفي الذكي التفاعلي (Interactive AI Q&A / RAG)
استجوب وثائق مشروعك باللغة الطبيعية! يقوم المساعد بالبحث الدلالي داخل ملفات Markdown والإجابة بدقة مع الاستشهاد بالمصادر دون أي تكلفة.

In [ ]:
#@title 8. اسأل مساعد التوثيق الذكي حول المشروع { display-mode: "form" }
QUESTION = "ما هي معمارية النظام والمكونات الرئيسية؟" #@param {type:"string"}
AUDIENCE = "Developer" #@param ["Developer", "Management", "End-User"]

chat_engine = pipeline.get_chat_engine()
chat_engine.load_documents()

print(f'🔍 جاري استرجاع المعلومات والإجابة عن: "{QUESTION}"...\n')
answer = chat_engine.ask(QUESTION, audience=AUDIENCE)
print('='*70)
print(answer)
print('='*70)